# 🧪 AI-Scientist v1 on Kaggle

## Fully Automated Scientific Discovery — Powered by DeepSeek V4 Flash 🚀

This notebook runs [SakanaAI's AI-Scientist v1](https://github.com/SakanaAI/AI-Scientist) — a fully automated scientific discovery pipeline.

### Pipeline overview:
1. 💡 **Idea Generation** — LLM generates novel research ideas
2. 🧪 **Experimentation** — Automated template-based experiments
3. 📝 **Write-up** — Auto-generates LaTeX paper
4. 🔍 **Review** — LLM reviews its own paper

> ⚠️ **Warning:** This pipeline executes LLM-generated code. Review outputs carefully. Use in a sandboxed environment.

> 💰 **Cost:** ~$0.50–3 per run with DeepSeek V4 Flash (10× cheaper than GPT-4o)

> ⚙️ **Kaggle setup:** Go to **Settings → Internet → Enable** (required for `pip install`). Select **GPU T4 x2** or **P100** accelerator.

## 🔑 Setup Instructions

### Add API keys as Kaggle Secrets
1. Open the **Add-ons** panel (🔑 key icon in right sidebar)
2. Click **Secrets** → **Add Secret**
3. Add each key with its exact name below

| Provider | Secret Name | Model Name | Cost |
|----------|-------------|------------|------|
| **DeepSeek** | `DEEPSEEK_API_KEY` | `deepseek-v4-flash` | ~$0.50–1/run 💥 |
| **DeepSeek** | `DEEPSEEK_API_KEY` | `deepseek-chat` | ~$0.50–1/run |
| **OpenAI** | `OPENAI_API_KEY` | `gpt-4o-mini` | ~$1–2/run |
| **Anthropic** | `ANTHROPIC_API_KEY` | `claude-3-5-sonnet-20241022` | ~$15/run |

### Optional: AndroZoo API (for Android Malware template)
- **`ANDROZOO_API_KEY`**: Get at [https://androzoo.uni.lu/api](https://androzoo.uni.lu/api)

DeepSeek V4 Flash (`deepseek-v4-flash`) is the most **cost-effective** option — get your key at:
👉 [https://platform.deepseek.com/api_keys](https://platform.deepseek.com/api_keys)

> If `deepseek-v4-flash` is not available, use `deepseek-chat` instead.

Secrets are loaded via `kaggle_secrets.UserSecretsClient()`.

In [ ]:
import os, sys, json
from pathlib import Path

WORKING_DIR = Path("/kaggle/working")
PROJECT_DIR = WORKING_DIR / "AI-Scientist"
os.chdir(WORKING_DIR)

# Clone your forked repo (includes deepseek-v4-flash support)
# Remove existing dir first so this cell is safe to re-run
if PROJECT_DIR.exists():
    !rm -rf "$PROJECT_DIR"
!git clone --depth 1 https://github.com/brahimje/AI-Scientist.git
%cd $PROJECT_DIR

# Install LaTeX (smaller package for Kaggle)
!apt-get update -qq && apt-get install -y -qq texlive-latex-base texlive-latex-extra texlive-fonts-recommended texlive-bibtex-extra chktex poppler-utils > /dev/null 2>&1

# Install Python packages (skip torch + comments — Kaggle already has torch)
!grep -vE '^#|^torch|^$' requirements.txt > /tmp/requirements_no_torch.txt
!pip install -q -r /tmp/requirements_no_torch.txt

# Verify GPU
import torch
if torch.cuda.is_available():
    print("✅ GPU:", torch.cuda.get_device_name(0))
    print("   VRAM: {:.1f} GB".format(torch.cuda.get_device_properties(0).total_memory / 1e9))
else:
    print("⚠️ No GPU detected — will use CPU")

In [ ]:
# Load API keys from Kaggle Secrets
# Add them in: Add-ons (right sidebar) → Secrets → Add Secret
# You only need to add the keys for the provider you want to use

def _get_secret(name: str) -> str:
    """Safely get a Kaggle secret, returning empty string if not set."""
    try:
        from kaggle_secrets import UserSecretsClient
        client = UserSecretsClient()
        return client.get_secret(name)
    except:
        return os.environ.get(name, "")

os.environ["OPENAI_API_KEY"] = _get_secret("OPENAI_API_KEY")
os.environ["ANTHROPIC_API_KEY"] = _get_secret("ANTHROPIC_API_KEY")
os.environ["DEEPSEEK_API_KEY"] = _get_secret("DEEPSEEK_API_KEY")
os.environ["S2_API_KEY"] = _get_secret("S2_API_KEY")
os.environ["ANDROZOO_API_KEY"] = _get_secret("ANDROZOO_API_KEY")

# Show which keys are set
for k in ["OPENAI_API_KEY", "ANTHROPIC_API_KEY", "DEEPSEEK_API_KEY", "S2_API_KEY", "ANDROZOO_API_KEY"]:
    status = "✅ Set" if os.environ.get(k) else "❌ Not set"
    print(f"   {status}: {k}")

In [ ]:
%cd $PROJECT_DIR

# ⚡ Choose your mode:
#   "quick"    → Use pretrained GPT-2 + fine-tune (~10 min, only shakespeare_char)
#   "full"     → Train from scratch (~6 hours, all datasets)
MODE = "quick"

if MODE == "quick":
    print("⚡ Quick mode: downloading pretrained GPT-2 + fine-tuning (~10 min)...")
    print("📦 Preparing NanoGPT data...")
    !python data/shakespeare_char/prepare.py
    %cd templates/nanoGPT
    !python experiment.py --out_dir run_0 --init_from pretrained --datasets shakespeare_char
    !python plot.py
    %cd $PROJECT_DIR
    print("✅ Quick baseline ready!")
else:
    print("📦 Preparing NanoGPT data (downloading ~100MB)...")
    !python data/enwik8/prepare.py
    !python data/shakespeare_char/prepare.py
    !python data/text8/prepare.py

    print("🏃 Running baseline experiment...")
    %cd templates/nanoGPT
    !python experiment.py --out_dir run_0 --init_from scratch
    !python plot.py
    %cd $PROJECT_DIR
    print("✅ NanoGPT baseline ready!")

## 🚀 How to Launch AI Scientist

The main launch command:

```bash
python launch_scientist.py --model "deepseek-v4-flash" --experiment nanoGPT_lite --num-ideas 2 --init-from pretrained
```

### Available templates:
| Template | Description | Kaggle-friendly |
|----------|-------------|:------:|
| `nanoGPT_lite` | Small GPT language model 🔤 | ✅ Yes |
| `android_malware` | Android malware detection 📱🛡️ | ✅ Yes (demo mode) |
| `nanoGPT` | Full GPT language model | ✅ Yes (P100/T4) |
| `2d_diffusion` | 2D diffusion denoising 🎨 | ⚠️ May be heavy |
| `grokking` | Algorithmic grokking 🧮 | ⚠️ May be heavy |

> 💡 **New!** `android_malware` uses AndroZoo API for Android malware detection — state-of-the-art topic!

### ⚡ Speed comparison (`--init-from`):

| Mode | Time per run | Total (5 runs × 2 ideas) |
|------|:------------:|:------------------------:|
| `--init-from pretrained` | **~10 min** | **~2 hours** ✅ |
| `--init-from scratch` | ~73 min | ~12 hours ⚠️ |

### Recommended models:
| Model | Quality | Cost |
|-------|---------|------|
| `deepseek-v4-flash` | Good | ~$0.50–1/run 💥 |
| `deepseek-chat` | Good | ~$0.50–1/run |
| `gpt-4o-mini` | Good | ~$1–2/run |
| `claude-3-5-sonnet-20241022` | Best | ~$15/run |

> For 2D Diffusion: first install NPEET (see Cell 9 tips)

In [ ]:
%cd $PROJECT_DIR

# ✏️ CONFIGURE YOUR RUN HERE
# DeepSeek is recommended as the default (cheapest, ~$0.50-1/run)
MODEL = "deepseek-v4-flash"      # Fallback: "deepseek-chat"
EXPERIMENT = "nanoGPT_lite"      # Template: nanoGPT_lite, android_malware, 2d_diffusion, grokking, nanoGPT
NUM_IDEAS = 2                    # Number of ideas to try (reduce to 1 if low on time)
INIT_FROM = "pretrained"         # "pretrained" (~10 min/run) or "scratch" (~73 min/run)

print("=" * 50)
print("🚀 Launching AI Scientist v1")
print("=" * 50)
print(f"   Model:      {MODEL}")
print(f"   Experiment: {EXPERIMENT}")
print(f"   Ideas:      {NUM_IDEAS}")
print(f"   Init:       {INIT_FROM} (pretrained ≈10 min/run, scratch ≈73 min/run)")
print()

!python launch_scientist.py \
    --model {MODEL} \
    --experiment {EXPERIMENT} \
    --num-ideas {NUM_IDEAS} \
    --init-from {INIT_FROM} \
    2>&1

print()
print("✅ AI Scientist run complete!")

In [ ]:
%cd $PROJECT_DIR

print("📂 Results:\n")

import os, glob
pdfs = sorted(glob.glob("**/*.pdf", recursive=True))
if pdfs:
    for pdf in pdfs:
        size = os.path.getsize(pdf) / 1e6
        print(f"   📄 {pdf} ({size:.1f} MB)")
else:
    print("   No PDFs found yet.")

# Find experiment folders
folders = sorted(glob.glob("templates/**/run_*", recursive=True))
if folders:
    print()
    for f in folders:
        print(f"   📁 {f}")

# Kaggle output is in /kaggle/working — files auto-save, download from Output tab
print("\n💡 Download: Go to the **Output** tab (top-right) → tick files → **Download**")

## 💡 Tips for Kaggle

### ⏰ Session limits
- Kaggle limits GPU sessions to **9 hours** (enough for pretrained mode!)
- Scratch mode (~12h) **will timeout** — use `--init-from pretrained`

### 🌐 Internet access
- **Settings → Internet → Enable** (required for `pip install` and HuggingFace downloads)
- Disabled by default — don't forget to turn it on!

### 💾 Saving outputs
- All files in `/kaggle/working/AI-Scientist/` are automatically saved
- Download from the **Output** tab (top-right toolbar)
- Or use: **Add-ons → Save as Version → Save Output Only**

### 📊 GPU options
- **Settings → Accelerator → GPU T4 x2** (recommended, 32GB VRAM)
- Or **GPU P100** (16GB VRAM, slightly slower)
- TPU is **not** supported (experiments use PyTorch CUDA)

### 🎨 2D Diffusion template
If using `2d_diffusion`, first run:
```python
!git clone https://github.com/gregversteeg/NPEET.git
%cd NPEET
!pip install .
%cd $PROJECT_DIR
```

### 🐘 CUDA Out of Memory
- Use **Lite** experiment variant (`nanoGPT_lite`)
- Reduce `NUM_IDEAS` to 1
- Restart session: **Settings → Restart & Clear Output**